# Landslide Network Economic Damage (Source Zones)

This notebook is fully separate from river/coastal workflows and does not modify existing files.

Method used here:
- Source-zone hazard where landslide probability `>= 0.5`
- Damage ratio `= 1.0` in source zones
- Same network-hazard intersection approach as `vector_raster_intersections.py`
- Damage calculated from existing cost columns in network layers


In [ ]:
from pathlib import Path
import subprocess

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Paths (kept separate from existing flood outputs)
ROOT = Path('/Users/robynhaggis/Documents/Geospatial_analysis')
BASE = ROOT / 'dphil_papers'
PAPER3 = BASE / 'dphil_paper_3'

LANDSLIDE_INPUT_DIR = PAPER3 / 'inputs' / 'landslides'
NETWORK_METADATA_CSV = BASE / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'
INTERSECTION_SCRIPT = BASE / 'robyns_libraries/vector_raster_intersections.py'

RESULTS_DIR = PAPER3 / 'results' / 'landslide_network_damage_analysis'
BINARY_HAZARD_DIR = RESULTS_DIR / 'hazard_binary_pge050'
INTERSECTIONS_DIR = RESULTS_DIR / 'network_hazard_intersections'
SUMMARY_DIR = RESULTS_DIR / 'damage_summaries'

for d in [RESULTS_DIR, BINARY_HAZARD_DIR, INTERSECTIONS_DIR, SUMMARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RESULTS_DIR

In [ ]:
# Scenario rasters
scenario_prob_rasters = {
    'baseline': LANDSLIDE_INPUT_DIR / 'landslide_probability_Baseline.tif',
    'deforestation': LANDSLIDE_INPUT_DIR / 'landslide_probability_Deforestation.tif',
    'reafforestation': LANDSLIDE_INPUT_DIR / 'landslide_probability_Reafforestation.tif',
}

for k, p in scenario_prob_rasters.items():
    print(f'{k:>14}: {p.name} | exists={p.exists()}')

In [ ]:
# Build binary source-zone hazard rasters (1 if probability >= 0.5, else 0)
THRESHOLD = 0.5
NODATA_OUT = 255

binary_rows = []
binary_rasters = {}

for scenario, src_path in scenario_prob_rasters.items():
    out_path = BINARY_HAZARD_DIR / f'landslide_source_pge050_{scenario}.tif'

    with rasterio.open(src_path) as src:
        arr = src.read(1)
        nodata_in = src.nodata

        valid = np.isfinite(arr)
        if nodata_in is not None and np.isfinite(nodata_in):
            valid &= arr != nodata_in

        binary = np.full(arr.shape, NODATA_OUT, dtype=np.uint8)
        binary[valid] = (arr[valid] >= THRESHOLD).astype(np.uint8)

        profile = src.profile.copy()
        profile.update(dtype='uint8', nodata=NODATA_OUT, count=1, compress='lzw')

        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(binary, 1)

        at_risk = int(np.sum(binary == 1))
        valid_count = int(np.sum(binary != NODATA_OUT))

        binary_rows.append({
            'scenario': scenario,
            'input_raster': str(src_path),
            'binary_raster': str(out_path),
            'crs': str(src.crs),
            'shape': str(src.shape),
            'threshold': THRESHOLD,
            'at_risk_pixels': at_risk,
            'valid_pixels': valid_count,
            'at_risk_pct': (100 * at_risk / valid_count) if valid_count else np.nan,
        })

        binary_rasters[scenario] = out_path

binary_summary = pd.DataFrame(binary_rows)
binary_summary.to_csv(SUMMARY_DIR / 'landslide_binary_hazard_summary.csv', index=False)
binary_summary

In [ ]:
# Prepare CSVs for vector-raster intersections (separate landslide run)
network_details = pd.read_csv(NETWORK_METADATA_CSV)

# Intersection script expects path relative to data root in config.json.
# Existing workflow adjusts networks/ -> networks/networks/.
network_paths = network_details[['path']].drop_duplicates().reset_index(drop=True)
network_paths['path'] = network_paths['path'].str.replace(r'^networks/', 'networks/networks/', regex=True)

network_csv_for_intersections = INTERSECTIONS_DIR / 'network_layers_for_intersections_landslide.csv'
network_paths.to_csv(network_csv_for_intersections, index=False)

hazard_rows = []
for scenario, rpath in binary_rasters.items():
    hazard_rows.append({
        'hazard': 'landslide_source',
        'scenario': scenario,
        'key': f'ls_source_pge05_{scenario}',
        'path': str(rpath),   # absolute path is acceptable
        'fname': str(rpath),  # required by vector_raster_intersections.py
    })

hazard_csv_for_intersections = INTERSECTIONS_DIR / 'landslide_hazard_rasters_for_intersections.csv'
hazard_table = pd.DataFrame(hazard_rows)
hazard_table.to_csv(hazard_csv_for_intersections, index=False)

print('Network CSV:', network_csv_for_intersections)
print('Hazard  CSV:', hazard_csv_for_intersections)
hazard_table

## Run Intersections

Set `RUN_INTERSECTIONS = True` to run the same splitting/intersection engine used in your coastal workflow.

Outputs are written to:
- `results/landslide_network_damage_analysis/network_hazard_intersections`


In [ ]:
RUN_INTERSECTIONS = False

if RUN_INTERSECTIONS:
    cmd = [
        'python',
        str(INTERSECTION_SCRIPT),
        str(network_csv_for_intersections),
        str(hazard_csv_for_intersections),
        str(INTERSECTIONS_DIR),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping intersections (RUN_INTERSECTIONS=False). Set True when ready.')

In [ ]:
# Check which expected intersection files exist
hazard_slug = hazard_csv_for_intersections.stem

expected = []
for row in network_details.itertuples(index=False):
    expected_path = INTERSECTIONS_DIR / f'{row.asset_gpkg}_splits__{hazard_slug}__{row.asset_layer}.geoparquet'
    expected.append({
        'asset_gpkg': row.asset_gpkg,
        'asset_layer': row.asset_layer,
        'intersection_file': str(expected_path),
        'exists': expected_path.exists(),
    })

intersection_status = pd.DataFrame(expected)
intersection_status.to_csv(SUMMARY_DIR / 'intersection_file_status.csv', index=False)
intersection_status

In [ ]:
# Damage calculations from intersections
# - Source zones: damage_ratio = 1.0
# - Costs come from each network layer's existing cost columns

DAMAGE_RATIO_SOURCE = 1.0
SCENARIO_KEYS = {
    'baseline': 'ls_source_pge05_baseline',
    'deforestation': 'ls_source_pge05_deforestation',
    'reafforestation': 'ls_source_pge05_reafforestation',
}


def normalize_currency(unit_value: str) -> str:
    u = str(unit_value).upper().strip()
    if 'USD' in u:
        return 'USD'
    if 'J$' in u or '$J' in u or 'JD' in u:
        return 'JMD'
    return 'UNKNOWN'


def quantity_from_geometry(gdf: gpd.GeoDataFrame, unit_series: pd.Series) -> np.ndarray:
    g = gdf.to_crs(3448)
    length_m = g.geometry.length.to_numpy()
    area_m2 = g.geometry.area.to_numpy()

    units = unit_series.fillna('').astype(str).str.lower()
    qty = np.ones(len(gdf), dtype=float)

    mask_m2 = units.str.contains('/m2')
    mask_km = units.str.contains('/km')
    mask_m = units.str.contains('/m') & ~mask_m2 & ~mask_km

    qty[mask_m2.to_numpy()] = area_m2[mask_m2.to_numpy()]
    qty[mask_km.to_numpy()] = length_m[mask_km.to_numpy()] / 1000.0
    qty[mask_m.to_numpy()] = length_m[mask_m.to_numpy()]

    return qty


damage_rows = []
missing_files = []

for row in network_details.itertuples(index=False):
    split_file = INTERSECTIONS_DIR / f'{row.asset_gpkg}_splits__{hazard_slug}__{row.asset_layer}.geoparquet'

    if not split_file.exists():
        missing_files.append(str(split_file))
        continue

    gdf = gpd.read_parquet(split_file)

    unit_col = row.asset_cost_unit_column if isinstance(row.asset_cost_unit_column, str) else None
    if not unit_col or unit_col not in gdf.columns:
        gdf['_unit_tmp_'] = ''
        unit_col = '_unit_tmp_'

    id_col = row.asset_id_column if isinstance(row.asset_id_column, str) and row.asset_id_column in gdf.columns else None

    cost_columns = {
        'min': row.asset_min_cost_column,
        'mean': row.asset_mean_cost_column,
        'max': row.asset_max_cost_column,
    }

    for scenario_name, scenario_key in SCENARIO_KEYS.items():
        if scenario_key not in gdf.columns:
            continue

        exposed = gdf[gdf[scenario_key] == 1].copy()
        if exposed.empty:
            for cost_level in ['min', 'mean', 'max']:
                damage_rows.append({
                    'scenario': scenario_name,
                    'sector': row.sector,
                    'asset_description': row.asset_description,
                    'asset_gpkg': row.asset_gpkg,
                    'asset_layer': row.asset_layer,
                    'cost_level': cost_level,
                    'currency': 'UNKNOWN',
                    'damaged_assets': 0,
                    'damaged_split_geometries': 0,
                    'direct_damage': 0.0,
                })
            continue

        quantity = quantity_from_geometry(exposed, exposed[unit_col])
        currency = normalize_currency(exposed[unit_col].dropna().astype(str).head(1).iloc[0] if len(exposed) else '')

        damaged_assets = int(exposed[id_col].nunique()) if id_col else int(len(exposed))
        damaged_split_geometries = int(len(exposed))

        for cost_level, cost_col in cost_columns.items():
            if not isinstance(cost_col, str) or cost_col not in exposed.columns:
                continue

            unit_cost = pd.to_numeric(exposed[cost_col], errors='coerce').fillna(0.0).to_numpy()
            direct_damage = float(np.sum(quantity * unit_cost * DAMAGE_RATIO_SOURCE))

            damage_rows.append({
                'scenario': scenario_name,
                'sector': row.sector,
                'asset_description': row.asset_description,
                'asset_gpkg': row.asset_gpkg,
                'asset_layer': row.asset_layer,
                'cost_level': cost_level,
                'currency': currency,
                'damaged_assets': damaged_assets,
                'damaged_split_geometries': damaged_split_geometries,
                'direct_damage': direct_damage,
            })

if missing_files:
    print('Missing intersection files (run intersections first):', len(missing_files))
    print("\n".join(missing_files[:8]))

damage_asset = pd.DataFrame(damage_rows)
if not damage_asset.empty:
    damage_asset.to_csv(SUMMARY_DIR / 'landslide_damage_by_asset_scenario_costlevel.csv', index=False)

    damage_currency = (
        damage_asset
        .groupby(['scenario', 'currency', 'cost_level'], as_index=False)['direct_damage']
        .sum()
        .sort_values(['currency', 'cost_level', 'scenario'])
    )
    damage_currency.to_csv(SUMMARY_DIR / 'landslide_damage_by_currency_scenario_costlevel.csv', index=False)
else:
    damage_currency = pd.DataFrame()

damage_currency.head(20)


In [ ]:
# Scenario deltas: deforestation increase vs baseline, reafforestation reduction vs baseline
if not damage_currency.empty:
    pivot = (
        damage_currency
        .pivot_table(index=['currency', 'cost_level'], columns='scenario', values='direct_damage', aggfunc='sum', fill_value=0.0)
        .reset_index()
    )

    for col in ['baseline', 'deforestation', 'reafforestation']:
        if col not in pivot.columns:
            pivot[col] = 0.0

    pivot['deforestation_increase_abs'] = pivot['deforestation'] - pivot['baseline']
    pivot['deforestation_increase_pct'] = np.where(
        pivot['baseline'] > 0,
        100.0 * (pivot['deforestation'] - pivot['baseline']) / pivot['baseline'],
        np.nan,
    )

    pivot['reafforestation_reduction_abs'] = pivot['baseline'] - pivot['reafforestation']
    pivot['reafforestation_reduction_pct'] = np.where(
        pivot['baseline'] > 0,
        100.0 * (pivot['baseline'] - pivot['reafforestation']) / pivot['baseline'],
        np.nan,
    )

    pivot.to_csv(SUMMARY_DIR / 'landslide_damage_scenario_deltas.csv', index=False)
    display(pivot)
else:
    print('No damage summary available yet (likely intersections not run).')

In [ ]:
# Quick plot (if results exist)
if not damage_currency.empty:
    plot_df = damage_currency[damage_currency['cost_level'] == 'mean'].copy()
    if not plot_df.empty:
        fig, ax = plt.subplots(figsize=(9, 5))
        for cur, cur_df in plot_df.groupby('currency'):
            cur_df = cur_df.sort_values('scenario')
            ax.plot(cur_df['scenario'], cur_df['direct_damage'], marker='o', label=cur)
        ax.set_title('Landslide Direct Damage by Scenario (Mean Cost)')
        ax.set_ylabel('Direct damage (native currency)')
        ax.set_xlabel('Scenario')
        ax.legend(title='Currency')
        plt.show()
else:
    print('No plot yet. Run intersections and damage cells first.')

## Outputs Written

- `results/landslide_network_damage_analysis/hazard_binary_pge050/*.tif`
- `results/landslide_network_damage_analysis/network_hazard_intersections/*.geoparquet`
- `results/landslide_network_damage_analysis/damage_summaries/*.csv`
